In [ ]:
# =========================================================
# Multi-product detrended monthly climatology of surface ocean pCO2
# Strict valid-range handling version
# Exclude CSIR-ML6 and JMA-MLR
# Final 6 products:
#   1) NIES-ML3
#   2) CMEMS-LSCEv2
#   3) OceanSODA-ETHZv2
#   4) SJTU-AVIT
#   5) LDEO-HPD Extended
#   6) LDEO-Residual
# =========================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", category=RuntimeWarning)
xr.set_options(keep_attrs=True)

# =========================================================
# 1. User configuration
# =========================================================
START = "1990-01-01"
END = "2020-12-01"         # inclusive monthly-start
REF_YEAR = 2005

MONTHS = pd.date_range(START, END, freq="MS")
NMON = len(MONTHS)         # 372

OUT_DIR = Path("/data/wang/Result_pCO2/Muit_Product_clim")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "multi_product_detrended_monthly_climatology_1990_2020_ref2005_6products_noCSIR_noJMA.nc"

TARGET_LAT = np.arange(-89.5, 90.0, 1.0, dtype=np.float64)   # 180
TARGET_LON = np.arange(0.5, 360.0, 1.0, dtype=np.float64)    # 360

# ---- valid range for pCO2-like variables ----
PCO2_MIN_VALID = 0.0
PCO2_MAX_VALID = 1500.0

# ---- valid range for SST used in fCO2 -> pCO2 conversion ----
SST_MIN_VALID = -5.0
SST_MAX_VALID = 45.0

# If True, pco2_prior_clim_mean is a robust trimmed mean:
# drop the highest two and lowest two valid product values when >=5 products are available.
# For cells/months with <5 valid products, fallback to ordinary nanmean.
USE_TRIMMED_MEAN_FOR_PRIOR_MEAN = True

# ---- product paths ----
NIES_FILE = Path("/data/wang/pCO2/1Product/NIES-ML3/nies.ml.1982-2024.ver.2025.0.nc")
CMEMS_DIR = Path("/data/wang/CMEMS/pco2")
OCEANSODA_FILES = [
    Path("/data/wang/pCO2/1Product/OceanSODA-ETHZ-v2/sfco2-1980s-8D_25km-OceanSODAETHZv2.2024r01.nc"),
    Path("/data/wang/pCO2/1Product/OceanSODA-ETHZ-v2/sfco2-1990s-8D_25km-OceanSODAETHZv2.2024r01.nc"),
    Path("/data/wang/pCO2/1Product/OceanSODA-ETHZ-v2/sfco2-2000s-8D_25km-OceanSODAETHZv2.2024r01.nc"),
    Path("/data/wang/pCO2/1Product/OceanSODA-ETHZ-v2/sfco2-2010s-8D_25km-OceanSODAETHZv2.2024r01.nc"),
    Path("/data/wang/pCO2/1Product/OceanSODA-ETHZ-v2/sfco2-2020s-8D_25km-OceanSODAETHZv2.2024r01.nc"),
]
SJTU_FILE = Path("/data/wang/pCO2/1Product/SJTU_AVIT_pCO2/SJTU_AVIT_1982-2023_v20250501.nc")
LDEO_FILE = Path("/data/wang/pCO2/1Product/LDEO-HPD fCO2, with Extended Temporal Coverage/LDEO-HPD_1959-2023.nc")
LDEO_RESIDUAL_FILE = Path("/data/wang/pCO2/1Product/LDEO fCO2 - Residual Method/LDEO-Residual_1982-2023.nc")

# ---- external SST ----
OISST_DIR = Path("/data/wang/NOAAOISST/monthly(caulcate)")

# ---- fCO2 -> pCO2 coefficients ----
A_COEF = 1.00436
B_COEF = 4.669e-5


# =========================================================
# 2. Utility functions
# =========================================================
def build_empty_monthly_cube():
    return np.full((NMON, 180, 360), np.nan, dtype=np.float32)


def month_file_yyyymm(ts: pd.Timestamp) -> str:
    return f"{ts.year:04d}{ts.month:02d}"


def build_period_lookup(time_values):
    dt = pd.to_datetime(time_values)
    periods = pd.PeriodIndex(dt, freq="M")
    return {p: i for i, p in enumerate(periods)}


def sanitize_pco2(arr):
    """
    Enforce valid range for pCO2-like variables:
      0 < value <= 1500
    Everything else -> NaN
    Works for any shape.
    """
    arr = np.asarray(arr, dtype=np.float32)
    valid = np.isfinite(arr) & (arr > PCO2_MIN_VALID) & (arr <= PCO2_MAX_VALID)
    out = np.where(valid, arr, np.nan).astype(np.float32)
    return out


def sanitize_sst(arr):
    """
    Remove unrealistic SST/fill values before fCO2 -> pCO2 conversion.
    """
    arr = np.asarray(arr, dtype=np.float32)
    valid = np.isfinite(arr) & (arr >= SST_MIN_VALID) & (arr <= SST_MAX_VALID)
    return np.where(valid, arr, np.nan).astype(np.float32)


def build_area_regrid_info(src_lats, src_lons):
    """
    General area-weighted mapping from source regular-ish grid
    to target 1-degree global center grid.
    """
    src_lats = np.asarray(src_lats, dtype=np.float64)
    src_lons = np.asarray(src_lons, dtype=np.float64)

    lat_order = np.argsort(src_lats)
    lats_sorted = src_lats[lat_order]

    lons_360 = np.mod(src_lons, 360.0)
    lon_order = np.argsort(lons_360)
    lons_sorted = lons_360[lon_order]

    lat_bin = np.floor(lats_sorted + 90.0).astype(int)
    lon_bin = np.floor(lons_sorted).astype(int)

    valid_lat = (lat_bin >= 0) & (lat_bin < 180)
    valid_lon = (lon_bin >= 0) & (lon_bin < 360)

    lat_bin_2d = np.repeat(lat_bin[:, None], len(lons_sorted), axis=1)
    lon_bin_2d = np.repeat(lon_bin[None, :], len(lats_sorted), axis=0)
    cell_id = (lat_bin_2d * 360 + lon_bin_2d).ravel()

    weights_2d = np.cos(np.deg2rad(np.repeat(lats_sorted[:, None], len(lons_sorted), axis=1)))
    weights = weights_2d.ravel()

    valid_2d = (
        np.repeat(valid_lat[:, None], len(lons_sorted), axis=1)
        & np.repeat(valid_lon[None, :], len(lats_sorted), axis=0)
    )
    valid_mask = valid_2d.ravel()

    return {
        "lat_order": lat_order,
        "lon_order": lon_order,
        "cell_id": cell_id.astype(np.int32),
        "weights": weights.astype(np.float64),
        "valid_mask": valid_mask,
    }


def regrid_2d_area_weighted_to_1deg(field2d, rg_info):
    """
    Area-weighted regridding to target 1° grid.
    Assumes invalid values are already NaN.
    """
    arr = np.asarray(field2d, dtype=np.float64)
    arr = arr[rg_info["lat_order"], :][:, rg_info["lon_order"]]

    flat = arr.ravel()
    valid = np.isfinite(flat) & rg_info["valid_mask"]

    out = np.full(180 * 360, np.nan, dtype=np.float64)
    if valid.any():
        num = np.bincount(
            rg_info["cell_id"][valid],
            weights=flat[valid] * rg_info["weights"][valid],
            minlength=180 * 360,
        ).astype(np.float64)
        den = np.bincount(
            rg_info["cell_id"][valid],
            weights=rg_info["weights"][valid],
            minlength=180 * 360,
        ).astype(np.float64)

        with np.errstate(invalid="ignore", divide="ignore"):
            out = num / den
        out[den == 0] = np.nan

    return out.reshape(180, 360).astype(np.float32)


def prepare_spatial_mapper(lats, lons, name, tol=1e-6):
    """
    If the grid already matches target 1° grid after sorting, use direct remap.
    Otherwise, use area-weighted regridding.
    """
    lats = np.asarray(lats, dtype=np.float64)
    lons_360 = np.mod(np.asarray(lons, dtype=np.float64), 360.0)

    lat_order = np.argsort(lats)
    lon_order = np.argsort(lons_360)

    lats_sorted = lats[lat_order]
    lons_sorted = lons_360[lon_order]

    if (
        len(lats_sorted) == 180
        and len(lons_sorted) == 360
        and np.nanmax(np.abs(lats_sorted - TARGET_LAT)) <= tol
        and np.nanmax(np.abs(lons_sorted - TARGET_LON)) <= tol
    ):
        return {
            "mode": "direct",
            "lat_order": lat_order,
            "lon_order": lon_order,
        }

    return {
        "mode": "regrid",
        "rg_info": build_area_regrid_info(lats, lons),
        "name": name,
    }


def map_2d_to_target(field2d, mapper):
    """
    Map 2D field to target 1° grid.
    field2d should already have invalid values set to NaN.
    """
    if mapper["mode"] == "direct":
        arr = np.asarray(field2d, dtype=np.float32)
        out = arr[mapper["lat_order"], :][:, mapper["lon_order"]].astype(np.float32)
        return out
    return regrid_2d_area_weighted_to_1deg(field2d, mapper["rg_info"])


def map_3d_to_target(cube3d, mapper):
    """
    Map 3D cube [time, lat, lon] to target 1° grid.
    cube3d should already have invalid values set to NaN.
    """
    cube3d = np.asarray(cube3d)

    if mapper["mode"] == "direct":
        return (
            cube3d[:, mapper["lat_order"], :][:, :, mapper["lon_order"]]
            .astype(np.float32)
        )

    out = np.full((cube3d.shape[0], 180, 360), np.nan, dtype=np.float32)
    for i in range(cube3d.shape[0]):
        out[i] = regrid_2d_area_weighted_to_1deg(cube3d[i], mapper["rg_info"])
    return out


def sfco2_to_spco2(sfco2, sst):
    """
    Convert fCO2/fCO2-like values to pCO2/spCO2 and re-apply valid range.

    Approximation used here:
        pCO2 = fCO2 * (1.00436 - 4.669e-5 * SST)

    SST must be in degC. The function masks unrealistic SST values first.
    Works for 2D or 3D arrays by broadcasting.
    """
    sfco2 = sanitize_pco2(sfco2)
    sst = sanitize_sst(sst)

    factor = A_COEF - B_COEF * sst
    out = sfco2 * factor
    out = np.where(np.isfinite(sfco2) & np.isfinite(sst) & np.isfinite(factor), out, np.nan).astype(np.float32)
    out = sanitize_pco2(out)
    return out


def summarize_valid(data_372, name):
    valid_counts = np.isfinite(data_372).sum(axis=(1, 2))
    print(f"[{name}] monthly valid-cell summary:")
    print(f"  min  = {int(np.nanmin(valid_counts))}")
    print(f"  max  = {int(np.nanmax(valid_counts))}")
    print(f"  mean = {float(np.nanmean(valid_counts)):.1f}")


def detrended_monthly_climatology(data_372, months, ref_year=2005):
    """
    data_372: [time, lat, lon] monthly series on 1° target grid, 1990-01 ... 2020-12

    For each calendar month:
      1) fit y = a*x + b
      2) adjust all years to ref_year
      3) average adjusted values

    Invalid pCO2 values are removed before and after climatology calculation.

    Returns:
      clim   [12, 180, 360]
      slope  [12, 180, 360]
      n_valid[12, 180, 360]
    """
    data_372 = sanitize_pco2(data_372).astype(np.float64)

    clim = np.full((12, 180, 360), np.nan, dtype=np.float32)
    slope = np.full((12, 180, 360), np.nan, dtype=np.float32)
    n_valid = np.zeros((12, 180, 360), dtype=np.int16)

    years_all = np.array([t.year for t in months], dtype=np.float64)
    months_all = np.array([t.month for t in months], dtype=np.int16)

    for m in range(1, 13):
        idx = np.where(months_all == m)[0]
        y = data_372[idx, :, :]
        x = years_all[idx].astype(np.float64)
        x3 = x[:, None, None]

        mask = np.isfinite(y)
        n = mask.sum(axis=0).astype(np.int16)
        n_valid[m - 1] = n

        y_masked = np.where(mask, y, np.nan)
        x_masked = np.where(mask, x3, np.nan)

        with np.errstate(invalid="ignore"):
            x_mean = np.nanmean(x_masked, axis=0)
            y_mean = np.nanmean(y_masked, axis=0)

            cov = np.nansum((x3 - x_mean) * (y - y_mean) * mask, axis=0)
            var = np.nansum(((x3 - x_mean) ** 2) * mask, axis=0)

            a = np.full((180, 360), 0.0, dtype=np.float64)
            good = (n >= 2) & np.isfinite(cov) & np.isfinite(var) & (var > 0)
            a[good] = cov[good] / var[good]

            y_adj = y - a[None, :, :] * (x3 - ref_year)
            c = np.nanmean(np.where(mask, y_adj, np.nan), axis=0)
            c[n == 0] = np.nan

        c = sanitize_pco2(c)
        clim[m - 1] = c.astype(np.float32)

        a[n == 0] = np.nan
        slope[m - 1] = a.astype(np.float32)

    return clim, slope, n_valid


def decade_key_from_year(year):
    if 1980 <= year <= 1989:
        return "sfco2-1980s-8D_25km-OceanSODAETHZv2.2024r01.nc"
    elif 1990 <= year <= 1999:
        return "sfco2-1990s-8D_25km-OceanSODAETHZv2.2024r01.nc"
    elif 2000 <= year <= 2009:
        return "sfco2-2000s-8D_25km-OceanSODAETHZv2.2024r01.nc"
    elif 2010 <= year <= 2019:
        return "sfco2-2010s-8D_25km-OceanSODAETHZv2.2024r01.nc"
    elif 2020 <= year <= 2029:
        return "sfco2-2020s-8D_25km-OceanSODAETHZv2.2024r01.nc"
    else:
        raise ValueError(year)


# =========================================================
# 3. OISST monthly SST -> 1°
# =========================================================
def load_oisst_monthly_1deg(months):
    out = build_empty_monthly_cube()
    missing_files = []
    rg_info = None

    for i, ts in enumerate(months):
        yyyymm = month_file_yyyymm(ts)
        fn = OISST_DIR / f"oisst-avhrr-v02r01.{yyyymm}.nc"
        if not fn.exists():
            missing_files.append(str(fn))
            continue

        ds = xr.open_dataset(fn, decode_times=True)
        try:
            da = ds["sst"].isel(time=0, zlev=0)
            lats = ds["lat"].values
            lons = ds["lon"].values

            if rg_info is None:
                rg_info = build_area_regrid_info(lats, lons)

            out[i] = sanitize_sst(regrid_2d_area_weighted_to_1deg(da.values, rg_info))
        finally:
            ds.close()

    print(f"[OISST] loaded valid values: {np.isfinite(out).sum():,}")
    if missing_files:
        print(f"[OISST] missing files: {len(missing_files)}")
        for s in missing_files[:5]:
            print("   ", s)

    return out


# =========================================================
# 4. Generic loader for monthly 1°/near-1° products
# =========================================================
def load_monthly_var_to_target(
    nc_file,
    var_name,
    months,
    lat_name="lat",
    lon_name="lon",
    time_name="time",
    decode_times=True,
    name_for_log=None,
    sanitize_as_pco2=True,
):
    """
    Read monthly variable, subset 1990-2020, map to target 1° grid.
    Used for products with regular monthly time dimension.
    If sanitize_as_pco2=True, source values outside (0, 1500] are removed
    BEFORE remapping.
    """
    if name_for_log is None:
        name_for_log = Path(nc_file).name

    ds = xr.open_dataset(nc_file, decode_times=decode_times)
    try:
        da = ds[var_name].sel({time_name: slice(START, "2020-12-31")})
        # Ensure the numerical cube is always [time, lat, lon].
        # This avoids silent errors if a product stores dimensions in a different order.
        da = da.transpose(time_name, lat_name, lon_name)
        times = pd.to_datetime(da[time_name].values)

        cube_raw = da.values.astype(np.float32)
        if sanitize_as_pco2:
            cube_raw = sanitize_pco2(cube_raw)

        mapper = prepare_spatial_mapper(ds[lat_name].values, ds[lon_name].values, name_for_log)
        cube_target = map_3d_to_target(cube_raw, mapper)

        if sanitize_as_pco2:
            cube_target = sanitize_pco2(cube_target)

        lookup = build_period_lookup(times)
        out = build_empty_monthly_cube()

        for i, ts in enumerate(months):
            p = ts.to_period("M")
            if p in lookup:
                out[i] = cube_target[lookup[p]]

        if sanitize_as_pco2:
            out = sanitize_pco2(out)

        return out
    finally:
        ds.close()


# =========================================================
# 5. Product loaders
# =========================================================
def load_nies_monthly_spco2(months, oisst_1deg):
    sf_372 = load_monthly_var_to_target(
        nc_file=NIES_FILE,
        var_name="sfco2",
        months=months,
        lat_name="lat",
        lon_name="lon",
        time_name="time",
        decode_times=True,
        name_for_log="NIES-ML3",
        sanitize_as_pco2=True,
    )
    sf_372 = sanitize_pco2(sf_372)
    out = sfco2_to_spco2(sf_372, oisst_1deg)
    return sanitize_pco2(out)


def load_cmems_monthly_spco2(months):
    out = build_empty_monthly_cube()
    rg_info = None
    missing_files = []

    for i, ts in enumerate(months):
        yyyymm = month_file_yyyymm(ts)
        fn = CMEMS_DIR / f"cmems_obs-mob_glo_bgc-car_my_irr-i_{yyyymm}.nc"
        if not fn.exists():
            missing_files.append(str(fn))
            continue

        ds = xr.open_dataset(fn, decode_times=True)
        try:
            da = ds["spco2"].isel(time=0)
            lats = ds["latitude"].values
            lons = ds["longitude"].values

            if rg_info is None:
                rg_info = build_area_regrid_info(lats, lons)

            arr2d = sanitize_pco2(da.values)
            out[i] = regrid_2d_area_weighted_to_1deg(arr2d, rg_info)
        finally:
            ds.close()

    out = sanitize_pco2(out)

    if missing_files:
        print(f"[CMEMS] missing files: {len(missing_files)}")
        for s in missing_files[:5]:
            print("   ", s)

    return out


def open_oceansoda_datasets():
    opened = {}
    for fn in OCEANSODA_FILES:
        if not fn.exists():
            print(f"[OceanSODA] missing decade file: {fn}")
            continue
        ds = xr.open_dataset(fn, decode_times=True)
        opened[fn.name] = ds
    return opened


def load_oceansoda_monthly_spco2(months, oisst_1deg):
    """
    Technical implementation:
      1) remove invalid fCO2 values at native 8-day scale
      2) take monthly mean of 8-day fCO2
      3) area-weight downsample to 1°
      4) convert fCO2 -> pCO2 using 1° OISST
      5) re-apply valid range
    """
    out = build_empty_monthly_cube()
    opened = open_oceansoda_datasets()
    rg_info = None

    try:
        for i, ts in enumerate(months):
            key = decade_key_from_year(ts.year)
            if key not in opened:
                continue

            ds = opened[key]

            sub = ds["sfco2"].where(
                (ds["time"].dt.year == ts.year) & (ds["time"].dt.month == ts.month),
                drop=True
            )
            if sub.sizes.get("time", 0) == 0:
                continue

            sub = xr.where((sub > PCO2_MIN_VALID) & (sub <= PCO2_MAX_VALID), sub, np.nan)
            sf_month = sub.mean(dim="time", skipna=True)

            lats = ds["lat"].values
            lons = ds["lon"].values

            if rg_info is None:
                rg_info = build_area_regrid_info(lats, lons)

            sf_1deg = regrid_2d_area_weighted_to_1deg(sf_month.values, rg_info)
            sf_1deg = sanitize_pco2(sf_1deg)

            out[i] = sfco2_to_spco2(sf_1deg, oisst_1deg[i])

        return sanitize_pco2(out)
    finally:
        for ds in opened.values():
            ds.close()


def load_sjtu_monthly_spco2(months):
    out = load_monthly_var_to_target(
        nc_file=SJTU_FILE,
        var_name="spco2",
        months=months,
        lat_name="lat",
        lon_name="lon",
        time_name="time",
        decode_times=True,
        name_for_log="SJTU-AViT",
        sanitize_as_pco2=True,
    )
    return sanitize_pco2(out)


def load_ldeo_monthly_spco2(months, oisst_1deg):
    sf_372 = load_monthly_var_to_target(
        nc_file=LDEO_FILE,
        var_name="sfco2",
        months=months,
        lat_name="lat",
        lon_name="lon",
        time_name="time",
        decode_times=True,
        name_for_log="LDEO-HPD-Extended",
        sanitize_as_pco2=True,
    )
    sf_372 = sanitize_pco2(sf_372)
    out = sfco2_to_spco2(sf_372, oisst_1deg)
    return sanitize_pco2(out)


def load_ldeo_residual_monthly_spco2(months, oisst_1deg):
    sf_372 = load_monthly_var_to_target(
        nc_file=LDEO_RESIDUAL_FILE,
        var_name="sfco2",
        months=months,
        lat_name="lat",
        lon_name="lon",
        time_name="time",
        decode_times=True,
        name_for_log="LDEO-Residual",
        sanitize_as_pco2=True,
    )
    sf_372 = sanitize_pco2(sf_372)
    out = sfco2_to_spco2(sf_372, oisst_1deg)
    return sanitize_pco2(out)



# =========================================================
# 6. Load all monthly products
# =========================================================
oisst_1deg = load_oisst_monthly_1deg(MONTHS)
summarize_valid(oisst_1deg, "OISST_1deg")

nies_372 = load_nies_monthly_spco2(MONTHS, oisst_1deg)
cmems_372 = load_cmems_monthly_spco2(MONTHS)
osoda_372 = load_oceansoda_monthly_spco2(MONTHS, oisst_1deg)
sjtu_372 = load_sjtu_monthly_spco2(MONTHS)
ldeo_372 = load_ldeo_monthly_spco2(MONTHS, oisst_1deg)
ldeo_res_372 = load_ldeo_residual_monthly_spco2(MONTHS, oisst_1deg)

# Final safety pass on source monthly stacks
nies_372 = sanitize_pco2(nies_372)
cmems_372 = sanitize_pco2(cmems_372)
osoda_372 = sanitize_pco2(osoda_372)
sjtu_372 = sanitize_pco2(sjtu_372)
ldeo_372 = sanitize_pco2(ldeo_372)
ldeo_res_372 = sanitize_pco2(ldeo_res_372)

summarize_valid(nies_372, "NIES-ML3")
summarize_valid(cmems_372, "CMEMS-LSCEv2")
summarize_valid(osoda_372, "OceanSODA-ETHZv2")
summarize_valid(sjtu_372, "SJTU-AViT")
summarize_valid(ldeo_372, "LDEO-HPD-Extended")
summarize_valid(ldeo_res_372, "LDEO-Residual")


# =========================================================
# 7. Detrended monthly climatology for each product
# =========================================================
clim_nies, slope_nies, n_nies = detrended_monthly_climatology(nies_372, MONTHS, ref_year=REF_YEAR)
clim_cmems, slope_cmems, n_cmems = detrended_monthly_climatology(cmems_372, MONTHS, ref_year=REF_YEAR)
clim_osoda, slope_osoda, n_osoda = detrended_monthly_climatology(osoda_372, MONTHS, ref_year=REF_YEAR)
clim_sjtu, slope_sjtu, n_sjtu = detrended_monthly_climatology(sjtu_372, MONTHS, ref_year=REF_YEAR)
clim_ldeo, slope_ldeo, n_ldeo = detrended_monthly_climatology(ldeo_372, MONTHS, ref_year=REF_YEAR)
clim_ldeo_res, slope_ldeo_res, n_ldeo_res = detrended_monthly_climatology(ldeo_res_372, MONTHS, ref_year=REF_YEAR)

# Final safety pass on product climatologies
clim_nies = sanitize_pco2(clim_nies)
clim_cmems = sanitize_pco2(clim_cmems)
clim_osoda = sanitize_pco2(clim_osoda)
clim_sjtu = sanitize_pco2(clim_sjtu)
clim_ldeo = sanitize_pco2(clim_ldeo)
clim_ldeo_res = sanitize_pco2(clim_ldeo_res)

print("Detrended monthly climatology completed.")


def nan_trimmed_mean_product_axis(stack, trim_each_tail=2):
    """
    Robust product-axis mean for stack [product, month, lat, lon].

    If at least 2*trim_each_tail+1 valid products are available, drop the lowest
    and highest `trim_each_tail` values before averaging. Otherwise, fallback to
    ordinary nanmean to avoid discarding all available products.

    With the present 6-product configuration, trim_each_tail=2 means that cells
    with >=5 valid products use the central one or two product values. This is a
    robust mean close to a product-axis median. Set
    USE_TRIMMED_MEAN_FOR_PRIOR_MEAN=False if an ordinary product mean is needed.
    """
    stack = np.asarray(stack, dtype=np.float32)
    sorted_vals = np.sort(stack, axis=0)  # NaNs are sorted to the end by NumPy
    n_valid = np.sum(np.isfinite(stack), axis=0)

    ordinary = np.nanmean(stack, axis=0).astype(np.float32)
    trimmed = np.full_like(ordinary, np.nan, dtype=np.float32)

    use_trim = n_valid >= (2 * trim_each_tail + 1)
    if np.any(use_trim):
        middle = sorted_vals[trim_each_tail:-trim_each_tail, :, :, :]
        with np.errstate(invalid="ignore"):
            trimmed_all = np.nanmean(middle, axis=0).astype(np.float32)
        trimmed[use_trim] = trimmed_all[use_trim]

    out = ordinary
    out[use_trim] = trimmed[use_trim]
    return sanitize_pco2(out)


# =========================================================
# 8. Multi-product fusion
# =========================================================
prod_names = [
    "NIES_ML3",
    "CMEMS_LSCEv2",
    "OceanSODA_ETHZv2",
    "SJTU_AVIT",
    "LDEO_HPD_Extended",
    "LDEO_Residual",
]

clim_stack = np.stack(
    [
        clim_nies,
        clim_cmems,
        clim_osoda,
        clim_sjtu,
        clim_ldeo,
        clim_ldeo_res,
    ],
    axis=0
).astype(np.float32)   # [product, month, lat, lon]

# Extra safety before fusion
clim_stack = sanitize_pco2(clim_stack)

with np.errstate(invalid="ignore"):
    pco2_prior_clim = np.nanmedian(clim_stack, axis=0).astype(np.float32)
    pco2_prior_clim_mean = (
        nan_trimmed_mean_product_axis(clim_stack, trim_each_tail=2)
        if USE_TRIMMED_MEAN_FOR_PRIOR_MEAN
        else np.nanmean(clim_stack, axis=0).astype(np.float32)
    )
    sigma_benchmark = np.nanstd(clim_stack, axis=0, ddof=0).astype(np.float32)
    n_products = np.sum(np.isfinite(clim_stack), axis=0).astype(np.int16)

pco2_prior_clim = sanitize_pco2(pco2_prior_clim)
pco2_prior_clim_mean = sanitize_pco2(pco2_prior_clim_mean)

# Structural spread is not meaningful when <2 products
sigma_benchmark[n_products < 2] = np.nan

print("Fusion completed.")
print("n_products stats:")
print("  min :", int(np.nanmin(n_products)))
print("  max :", int(np.nanmax(n_products)))
print("  mean:", float(np.nanmean(n_products)))


# =========================================================
# 9. Save to NetCDF
# =========================================================
month_coord = np.arange(1, 13, dtype=np.int16)

ds_out = xr.Dataset(
    coords={
        "month": ("month", month_coord, {
            "long_name": "calendar month",
            "units": "1-12"
        }),
        "lat": ("lat", TARGET_LAT.astype(np.float32), {
            "long_name": "latitude",
            "units": "degrees_north",
            "axis": "Y"
        }),
        "lon": ("lon", TARGET_LON.astype(np.float32), {
            "long_name": "longitude",
            "units": "degrees_east",
            "axis": "X"
        }),
    },
    data_vars={
        # ---- product climatologies ----
        "clim_NIES_ML3": (("month", "lat", "lon"), clim_nies, {
            "long_name": "Detrended monthly climatology of NIES-ML3 pCO2 (converted from fCO2 using external OISST)",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "clim_CMEMS_LSCEv2": (("month", "lat", "lon"), clim_cmems, {
            "long_name": "Detrended monthly climatology of CMEMS-LSCEv2 pCO2",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "clim_OceanSODA_ETHZv2": (("month", "lat", "lon"), clim_osoda, {
            "long_name": "Detrended monthly climatology of OceanSODA-ETHZv2 pCO2 (converted from fCO2 using external OISST)",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "clim_SJTU_AVIT": (("month", "lat", "lon"), clim_sjtu, {
            "long_name": "Detrended monthly climatology of SJTU-AViT pCO2",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "clim_LDEO_HPD_Extended": (("month", "lat", "lon"), clim_ldeo, {
            "long_name": "Detrended monthly climatology of LDEO-HPD extended product pCO2 (converted from fCO2 using external OISST)",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "clim_LDEO_Residual": (("month", "lat", "lon"), clim_ldeo_res, {
            "long_name": "Detrended monthly climatology of LDEO-Residual pCO2 (converted from fCO2 using external OISST)",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),

        # ---- fused fields ----
        "pco2_prior_clim": (("month", "lat", "lon"), pco2_prior_clim, {
            "long_name": "Multi-product detrended monthly climatology prior (median across available products)",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "pco2_prior_clim_mean": (("month", "lat", "lon"), pco2_prior_clim_mean, {
            "long_name": "Multi-product detrended monthly climatology prior (robust trimmed mean or ordinary mean across available products)",
            "units": "microatm",
            "valid_range_note": "0 < value <= 1500"
        }),
        "sigma_benchmark": (("month", "lat", "lon"), sigma_benchmark, {
            "long_name": "Structural benchmark spread across product climatologies (std across available products)",
            "units": "microatm"
        }),
        "n_products": (("month", "lat", "lon"), n_products, {
            "long_name": "Number of valid products contributing to fusion at each grid cell and month",
            "units": "count"
        }),

        # ---- slopes ----
        "slope_NIES_ML3": (("month", "lat", "lon"), slope_nies, {
            "long_name": "Linear trend slope used in detrending for NIES-ML3",
            "units": "microatm yr-1"
        }),
        "slope_CMEMS_LSCEv2": (("month", "lat", "lon"), slope_cmems, {
            "long_name": "Linear trend slope used in detrending for CMEMS-LSCEv2",
            "units": "microatm yr-1"
        }),
        "slope_OceanSODA_ETHZv2": (("month", "lat", "lon"), slope_osoda, {
            "long_name": "Linear trend slope used in detrending for OceanSODA-ETHZv2",
            "units": "microatm yr-1"
        }),
        "slope_SJTU_AVIT": (("month", "lat", "lon"), slope_sjtu, {
            "long_name": "Linear trend slope used in detrending for SJTU-AViT",
            "units": "microatm yr-1"
        }),
        "slope_LDEO_HPD_Extended": (("month", "lat", "lon"), slope_ldeo, {
            "long_name": "Linear trend slope used in detrending for LDEO-HPD extended product",
            "units": "microatm yr-1"
        }),
        "slope_LDEO_Residual": (("month", "lat", "lon"), slope_ldeo_res, {
            "long_name": "Linear trend slope used in detrending for LDEO-Residual product",
            "units": "microatm yr-1"
        }),
    }
)

ds_out.attrs = {
    "title": "Six-product detrended monthly climatology of surface ocean pCO2",
    "summary": (
        "For each retained product, each grid cell, and each calendar month, a linear trend was fitted "
        "over 1990-2020, values were adjusted to reference year 2005, and the adjusted monthly "
        "values were averaged to form a detrended monthly climatology. The fused prior climatology "
        "includes the median and a robust mean across available products. CSIR-ML6 and JMA-MLR are "
        "excluded by design. sigma_benchmark is the standard deviation across available products. "
        "For all pCO2-like variables, only values within 0 < value <= 1500 are retained; all other "
        "values are treated as invalid."
    ),
    "reference_year": str(REF_YEAR),
    "climatology_period_start": "1990-01",
    "climatology_period_end": "2020-12",
    "target_grid": "1x1 degree global grid, lat=-89.5..89.5, lon=0.5..359.5",
    "valid_pco2_range": "0 < value <= 1500",
    "excluded_products": "CSIR-ML6, JMA-MLR",
    "products": ", ".join(prod_names),
    "fco2_to_pco2_formula": "pCO2 = fCO2 * (1.00436 - 4.669e-5 * SST)",
    "fco2_to_pco2_sst_source": str(OISST_DIR),
    "fusion_median_variable": "pco2_prior_clim",
    "fusion_mean_variable": "pco2_prior_clim_mean",
    "fusion_mean_note": "If USE_TRIMMED_MEAN_FOR_PRIOR_MEAN=True, this field drops the highest two and lowest two valid product climatologies where at least five products are available; otherwise it falls back to ordinary nanmean.",
    "note_1": "NIES-ML3, OceanSODA-ETHZv2, LDEO-HPD Extended, and LDEO-Residual are converted from fCO2/sfCO2 to pCO2 using external NOAA OISST monthly SST.",
    "note_2": "OceanSODA-ETHZv2 uses monthly mean of 8-day product, then area-weighted regridding to 1 degree, then fCO2/sfCO2 to pCO2 conversion with 1 degree OISST.",
    "note_3": "sigma_benchmark is set to NaN where fewer than 2 products are available.",
    "output_directory": str(OUT_DIR),
}

fillv = np.float32(9.96921e36)
encoding = {}

for v in ds_out.data_vars:
    if v == "n_products":
        encoding[v] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "int16",
            "_FillValue": -32767,
        }
    else:
        encoding[v] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32",
            "_FillValue": fillv,
        }

encoding["lat"] = {"dtype": "float32"}
encoding["lon"] = {"dtype": "float32"}
encoding["month"] = {"dtype": "int16"}

ds_out.to_netcdf(OUT_FILE, format="NETCDF4", encoding=encoding)

print("\nSaved to:")
print(OUT_FILE)

print("\nOutput variables:")
for v in ds_out.data_vars:
    print(" -", v)

# Final diagnostic check for pCO2-related outputs
check_vars = [
    "clim_NIES_ML3",
    "clim_CMEMS_LSCEv2",
    "clim_OceanSODA_ETHZv2",
    "clim_SJTU_AVIT",
    "clim_LDEO_HPD_Extended",
    "clim_LDEO_Residual",
    "pco2_prior_clim",
    "pco2_prior_clim_mean",
]

print("\nFinal validity check (should all be True):")
for v in check_vars:
    arr = ds_out[v].values
    bad = np.isfinite(arr) & ((arr <= PCO2_MIN_VALID) | (arr > PCO2_MAX_VALID))
    print(f"{v}: {not bad.any()}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
为 SOCAT_TRAIN.csv 新增/覆盖以下字段：
1) pco2_prior_clim_mapped
2) pCO2_prior
3) r_star_T

公式：
    pCO2_prior = pco2_prior_clim_mapped + delta_pCO2_T
    r_star_T   = pCO2 - pCO2_prior
"""


from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import xarray as xr


# =========================================================
# 1. User configuration
# =========================================================
CSV_FILE = Path("/data/wang/Result_pCO2/allpco2/SOCAT_TRAIN.csv")
NC_FILE = Path("/data/wang/Result_pCO2/Muit_Product_clim/multi_product_detrended_monthly_climatology_1990_2020_ref2005_6products_noCSIR_noJMA.nc")

NC_VAR = "pco2_prior_clim"

CSV_COL_MONTH = "Month"
CSV_COL_LAT = "Latitude"
CSV_COL_LON = "Longitude"
CSV_COL_DELTA_T = "delta_pCO2_T"
CSV_COL_PCO2 = "pCO2"

OUT_COL_CLIM = "pco2_prior_clim_mapped"
OUT_COL_PRIOR = "pCO2_prior"
OUT_COL_RSTAR = "r_star_T"

OVERWRITE_INPUT = True
MAKE_BACKUP = True

# Numerical tolerance
TOL_LAT = 1e-6
TOL_LON = 1e-6


# =========================================================
# 2. Utility functions
# =========================================================
def require_columns(df: pd.DataFrame, cols: list[str], file_desc: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{file_desc} 缺少必要字段: {missing}")


def normalize_lon_360(lon):
    """将经度统一到 [0, 360)。"""
    lon = np.asarray(lon, dtype=np.float64)
    return np.mod(lon, 360.0)


def build_month_indexer(month_values):
    """
    month_values should represent 1..12
    返回 {month_number: index}
    """
    months = np.asarray(month_values).astype(int)
    out = {}
    for i, m in enumerate(months):
        if 1 <= m <= 12:
            out[int(m)] = i
    return out


def check_monotonic_increasing(arr, name):
    arr = np.asarray(arr, dtype=np.float64)
    dif = np.diff(arr)
    if not np.all(dif > 0):
        raise ValueError(f"{name} 不是严格递增，当前脚本要求坐标单调递增。")


def nearest_index(sorted_array: np.ndarray, values: np.ndarray) -> np.ndarray:
    """
    对严格递增的一维数组 sorted_array，返回每个 values 的最近邻索引。
    """
    idx = np.searchsorted(sorted_array, values)
    idx = np.clip(idx, 0, len(sorted_array) - 1)

    left = np.clip(idx - 1, 0, len(sorted_array) - 1)
    right = idx

    choose_left = np.abs(values - sorted_array[left]) <= np.abs(values - sorted_array[right])
    out = np.where(choose_left, left, right)
    return out.astype(np.int64)


def backup_file(path: Path) -> Path:
    bak = path.with_suffix(path.suffix + ".bak")
    shutil.copy2(path, bak)
    return bak


# =========================================================
# 3. Main workflow
# =========================================================
def main():
    if not CSV_FILE.exists():
        raise FileNotFoundError(f"CSV 文件不存在: {CSV_FILE}")
    if not NC_FILE.exists():
        raise FileNotFoundError(f"NetCDF 文件不存在: {NC_FILE}")

    print("=" * 80)
    print("Input configuration")
    print("=" * 80)
    print(f"CSV_FILE        : {CSV_FILE}")
    print(f"NC_FILE         : {NC_FILE}")
    print(f"NC_VAR          : {NC_VAR}")
    print("Prior source    : 6 products, excluding CSIR-ML6 and JMA-MLR")
    print(f"OVERWRITE_INPUT : {OVERWRITE_INPUT}")
    print(f"MAKE_BACKUP     : {MAKE_BACKUP}")

    # -----------------------------------------------------
    # Read CSV
    # -----------------------------------------------------
    df = pd.read_csv(CSV_FILE)
    require_columns(
        df,
        [CSV_COL_MONTH, CSV_COL_LAT, CSV_COL_LON, CSV_COL_DELTA_T, CSV_COL_PCO2],
        "CSV"
    )

    # Force numeric conversion
    df[CSV_COL_MONTH] = pd.to_numeric(df[CSV_COL_MONTH], errors="coerce")
    df[CSV_COL_LAT] = pd.to_numeric(df[CSV_COL_LAT], errors="coerce")
    df[CSV_COL_LON] = pd.to_numeric(df[CSV_COL_LON], errors="coerce")
    df[CSV_COL_DELTA_T] = pd.to_numeric(df[CSV_COL_DELTA_T], errors="coerce")
    df[CSV_COL_PCO2] = pd.to_numeric(df[CSV_COL_PCO2], errors="coerce")

    # -----------------------------------------------------
    # Read NC
    # -----------------------------------------------------
    ds = xr.open_dataset(NC_FILE, decode_times=True)
    try:
        if NC_VAR not in ds.variables:
            raise KeyError(f"NetCDF 中未找到变量: {NC_VAR}")

        # Print product information from the prior climatology file to avoid accidentally reading the old 8-product file.
        products_attr = ds.attrs.get("products", "UNKNOWN")
        title_attr = ds.attrs.get("title", "UNKNOWN")
        print("=" * 80)
        print("NetCDF prior metadata")
        print("=" * 80)
        print(f"title    : {title_attr}")
        print(f"products : {products_attr}")

        da = ds[NC_VAR]

        # Basic dimension checks
        expected_dims = {"month", "lat", "lon"}
        if not expected_dims.issubset(set(da.dims)):
            raise ValueError(
                f"{NC_VAR} 需要包含维度 {expected_dims}，当前维度为: {da.dims}"
            )

        nc_month = ds["month"].values
        nc_lat = ds["lat"].values.astype(np.float64)
        nc_lon = ds["lon"].values.astype(np.float64)

        check_monotonic_increasing(nc_lat, "NC lat")
        check_monotonic_increasing(nc_lon, "NC lon")

        month_to_idx = build_month_indexer(nc_month)

        if set(range(1, 13)) - set(month_to_idx.keys()):
            raise ValueError(
                f"NC month 坐标不完整，缺少月份: {sorted(set(range(1,13)) - set(month_to_idx.keys()))}"
            )

        # Read the target variable as a NumPy array
        # shape expected: [month, lat, lon]
        clim = da.transpose("month", "lat", "lon").values.astype(np.float32)

        if "n_products" in ds.variables:
            nprod = ds["n_products"].transpose("month", "lat", "lon").values
            finite_nprod = nprod[np.isfinite(nprod)]
            if finite_nprod.size > 0:
                print(f"n_products range: {int(np.nanmin(finite_nprod))} - {int(np.nanmax(finite_nprod))}")

    finally:
        ds.close()

    # -----------------------------------------------------
    # Build nearest-neighbor mapping
    # -----------------------------------------------------
    month_vals = df[CSV_COL_MONTH].to_numpy()
    lat_vals = df[CSV_COL_LAT].to_numpy(dtype=np.float64)
    lon_vals = normalize_lon_360(df[CSV_COL_LON].to_numpy(dtype=np.float64))

    mapped = np.full(len(df), np.nan, dtype=np.float32)

    valid_basic = (
        np.isfinite(month_vals) &
        np.isfinite(lat_vals) &
        np.isfinite(lon_vals)
    )

    month_int = np.full(len(df), -9999, dtype=np.int32)
    month_int[valid_basic] = month_vals[valid_basic].astype(np.int32)

    valid_month = valid_basic & np.isin(month_int, np.arange(1, 13))
    valid_all = valid_month

    print("=" * 80)
    print("Mapping diagnostics")
    print("=" * 80)
    print(f"Total rows                : {len(df):,}")
    print(f"Rows with valid month/lat/lon : {int(valid_all.sum()):,}")

    if valid_all.any():
        lat_idx = nearest_index(nc_lat, lat_vals[valid_all])
        lon_idx = nearest_index(nc_lon, lon_vals[valid_all])
        mon_idx = np.array([month_to_idx[int(m)] for m in month_int[valid_all]], dtype=np.int64)

        mapped_vals = clim[mon_idx, lat_idx, lon_idx]
        mapped[valid_all] = mapped_vals.astype(np.float32)

    # Write or overwrite pco2_prior_clim_mapped
    df[OUT_COL_CLIM] = mapped

    # -----------------------------------------------------
    # Compute pCO2_prior and r_star_T
    # -----------------------------------------------------
    df[OUT_COL_PRIOR] = df[OUT_COL_CLIM] + df[CSV_COL_DELTA_T]
    df[OUT_COL_RSTAR] = df[CSV_COL_PCO2] - df[OUT_COL_PRIOR]

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------
    n_clim = int(np.isfinite(df[OUT_COL_CLIM]).sum())
    n_prior = int(np.isfinite(df[OUT_COL_PRIOR]).sum())
    n_rstar = int(np.isfinite(df[OUT_COL_RSTAR]).sum())

    print("=" * 80)
    print("Output diagnostics")
    print("=" * 80)
    print(f"Finite {OUT_COL_CLIM:>24s}: {n_clim:,}")
    print(f"Finite {OUT_COL_PRIOR:>24s}: {n_prior:,}")
    print(f"Finite {OUT_COL_RSTAR:>24s}: {n_rstar:,}")

    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------
    if MAKE_BACKUP and OVERWRITE_INPUT:
        bak = backup_file(CSV_FILE)
        print(f"Backup created: {bak}")

    if OVERWRITE_INPUT:
        out_csv = CSV_FILE
    else:
        out_csv = CSV_FILE.with_name(CSV_FILE.stem + "_with_prior_rstar.csv")

    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"Saved CSV: {out_csv}")

    print("=" * 80)
    print("Done")
    print("=" * 80)


if __name__ == "__main__":
    main()